# CheckThat Task 3 — Stage 3 Verdict-Aware Evidence Packet Builder

**Version:** `PY_PACKET_v1`

This notebook reads Stage 2 outputs and builds compact claim-level evidence packets for Stage 4.

Stage 3 is **Python-only**:

- no LLM calls
- uses `original_rating` if available
- promotes only high-value evidence to `must_cite_evidence`
- keeps packets compact: usually 3–7 must-cite items
- saves resumable JSON to Drive

**Main output:**

```text
outputs/stage3/stage3_packets_TARGET10.json
```

## 1. Mount Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


## 2. Configuration

Edit this cell if your Drive folder, dataset tag, or target IDs change.

In [2]:
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/CheckThat_Task3_Dataset")

# Examples: "TARGET10", "100_test", "train", "dev"
DATASET_TAG = "test_dataset"

# Stage 2 input candidates.
# First name matches the new design; second supports older naming.
STAGE2_IN_CANDIDATES = [
    DRIVE_BASE / f"outputs/stage2/stage2_extract_{DATASET_TAG}.json",
    DRIVE_BASE / f"outputs/stage2/stage2_api_extract_{DATASET_TAG}.json",
]

# Stage 0 is used only to recover claim text/original rating if Stage 2 lacks them.
STAGE0_IN_CANDIDATES = [
    DRIVE_BASE / f"outputs/stage0/flattened_sources_{DATASET_TAG}.json",
]

# Optional claim-level metadata candidates. Add your exact file here if needed.
CLAIM_METADATA_CANDIDATES = [
    DRIVE_BASE / f"claims_{DATASET_TAG}.json",
    DRIVE_BASE / f"metadata_{DATASET_TAG}.json",
    DRIVE_BASE / f"dataset_{DATASET_TAG}.json",
    DRIVE_BASE / "claims.json",
    DRIVE_BASE / "metadata.json",
    DRIVE_BASE / "dataset.json",
]

LOCAL_STAGE3_DIR = Path("/content/outputs/stage3")
LOCAL_STAGE3_DIR.mkdir(parents=True, exist_ok=True)
STAGE3_OUT = LOCAL_STAGE3_DIR / f"stage3_packets_{DATASET_TAG}.json"

DRIVE_STAGE3_DIR = DRIVE_BASE / "outputs/stage3"
DRIVE_STAGE3_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_STAGE3_OUT = DRIVE_STAGE3_DIR / STAGE3_OUT.name

# Your requested TARGET10 debug set: 30349 to 30358 inclusive.
TARGET_CLAIM_IDS = set()
ENFORCE_TARGET_CLAIM_IDS = False

MIN_MUST_CITE = 2
MAX_MUST_CITE = 10
MAX_OPTIONAL_CONTEXT = 4
MAX_SNIPPETS_PER_EVIDENCE = 5

print("Dataset tag:", DATASET_TAG)
print("Drive base:", DRIVE_BASE)
print("Stage 3 local output:", STAGE3_OUT)
print("Stage 3 Drive output:", DRIVE_STAGE3_OUT)

Dataset tag: test_dataset
Drive base: /content/drive/MyDrive/CheckThat_Task3_Dataset
Stage 3 local output: /content/outputs/stage3/stage3_packets_test_dataset.json
Stage 3 Drive output: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage3/stage3_packets_test_dataset.json


## 3. Utilities

In [3]:
import json
import re
import shutil
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path


def load_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None


def clean_text(x, max_len=None):
    s = re.sub(r"\s+", " ", str(x or "")).strip()
    if max_len and len(s) > max_len:
        s = s[:max_len].rstrip() + "..."
    return s


def safe_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default


def claim_id(x):
    return str(x.get("id", "")).strip()


def source_key(x):
    return f'{str(x.get("id", ""))}||{x.get("file_name", "")}||{x.get("url", "")}'


def normalize_rating(value):
    raw = clean_text(value).lower().replace("-", "_").replace(" ", "_")
    raw = re.sub(r"_+", "_", raw)

    aliases = {
        "true": "true",
        "correct": "true",
        "mostly_true": "mostly_true",
        "partly_true": "half_true",
        "half_true": "half_true",
        "mixture": "half_true",
        "mixed": "half_true",
        "misleading": "misleading",
        "false": "false",
        "incorrect": "false",
        "pants_on_fire": "pants_fire",
        "pants_fire": "pants_fire",
        "fake": "pants_fire",
        "satire": "satire",
        "satirical": "satire",
    }
    return aliases.get(raw, raw or "unknown")


def listify(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

## 4. Verify inputs and load data

In [4]:
STAGE2_IN = first_existing(STAGE2_IN_CANDIDATES)

print("Stage 2 input candidates:")
for p in STAGE2_IN_CANDIDATES:
    print(" -", p, "exists=", p.exists())

if STAGE2_IN is None:
    raise FileNotFoundError(
        "No Stage 2 file found. Expected one of:\n" +
        "\n".join(str(p) for p in STAGE2_IN_CANDIDATES)
    )

stage2_rows = load_json(STAGE2_IN)

if ENFORCE_TARGET_CLAIM_IDS:
    stage2_rows = [x for x in stage2_rows if claim_id(x) in TARGET_CLAIM_IDS]

print("\nSelected Stage 2 input:", STAGE2_IN)
print("Stage 2 rows loaded:", len(stage2_rows))
print("Claim IDs in Stage 2:", sorted({claim_id(x) for x in stage2_rows}))

if ENFORCE_TARGET_CLAIM_IDS:
    found_ids = {claim_id(x) for x in stage2_rows}
    missing_ids = sorted(TARGET_CLAIM_IDS - found_ids)
    if missing_ids:
        raise ValueError(f"Missing required target IDs from Stage 2: {missing_ids}")


Stage 2 input candidates:
 - /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage2/stage2_extract_test_dataset.json exists= True
 - /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage2/stage2_api_extract_test_dataset.json exists= False

Selected Stage 2 input: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage2/stage2_extract_test_dataset.json
Stage 2 rows loaded: 6561
Claim IDs in Stage 2: ['0_ambigsnopes', '100_ambigsnopes', '104_ambigsnopes', '106_ambigsnopes', '107_ambigsnopes', '109_ambigsnopes', '111_ambigsnopes', '113_ambigsnopes', '114_ambigsnopes', '116_ambigsnopes', '117_ambigsnopes', '118_ambigsnopes', '11_ambigsnopes', '120_ambigsnopes', '122_ambigsnopes', '124_ambigsnopes', '127_ambigsnopes', '128_ambigsnopes', '129_ambigsnopes', '12_ambigsnopes', '130_ambigsnopes', '131_ambigsnopes', '133_ambigsnopes', '134_ambigsnopes', '135_ambigsnopes', '136_ambigsnopes', '139_ambigsnopes', '13_ambigsnopes', '140_ambigsnopes', '142_ambigsnopes', '144_ambigsno

In [5]:
from collections import Counter

print("Stage 2 file used:", STAGE2_IN)
print("Stage 2 rows:", len(stage2_rows))
print("Unique claim IDs:", len({claim_id(x) for x in stage2_rows}))

print("\nSource relations:")
print(Counter(x.get("source_relation") for x in stage2_rows))

print("\nEvidence roles:")
print(Counter(x.get("evidence_role") for x in stage2_rows))

print("\nkeep_for_writing:")
print(Counter(bool(x.get("keep_for_writing")) for x in stage2_rows))

assert len(stage2_rows) == 6561, f"Expected repaired Stage 2 rows around 6561, got {len(stage2_rows)}"

Stage 2 file used: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage2/stage2_extract_test_dataset.json
Stage 2 rows: 6561
Unique claim IDs: 1146

Source relations:
Counter({'useful_context': 2251, 'direct_evidence': 2103, 'off_topic': 986, 'claim_origin': 939, 'no_content': 282})

Evidence roles:
Counter({'background_context': 1485, 'off_topic': 986, 'claim_origin': 883, 'core_refutation': 711, 'statistical_evidence': 648, 'core_support': 524, 'legal_or_definition_context': 341, 'no_content': 282, 'scientific_context': 179, 'rating_caveat': 170, 'visual_misattribution': 112, 'visual_origin': 102, 'temporal_refutation': 75, 'satirical_origin': 36, 'fake_news_origin': 20, 'image_manipulation': 7})

keep_for_writing:
Counter({True: 4873, False: 1688})


## 5. Load claim metadata and original ratings

Stage 3 may use the original rating. This cell tries to recover it from Stage 0 or optional metadata files.

In [6]:
RATING_FIELDS = [
    "original_rating", "rating", "veracity", "label", "truth_label",
    "claim_rating", "final_rating", "fact_check_rating", "normalized_rating",
]

CLAIM_FIELDS = ["claim", "claim_text", "text", "statement"]


def extract_claim_metadata_from_records(records):
    meta = {}
    if isinstance(records, dict):
        # Claim-level dict: {id: {...}}
        possible_items = []
        for k, v in records.items():
            if isinstance(v, dict):
                item = dict(v)
                item.setdefault("id", k)
                possible_items.append(item)
            else:
                possible_items.append({"id": k, "claim": v})
        records = possible_items

    if not isinstance(records, list):
        return meta

    for row in records:
        if not isinstance(row, dict):
            continue
        cid = claim_id(row)
        if not cid:
            continue

        if cid not in meta:
            meta[cid] = {"id": cid, "claim": "", "original_rating": "unknown"}

        for f in CLAIM_FIELDS:
            if row.get(f) and not meta[cid].get("claim"):
                meta[cid]["claim"] = clean_text(row.get(f))

        for f in RATING_FIELDS:
            if row.get(f) and meta[cid].get("original_rating") == "unknown":
                meta[cid]["original_rating"] = clean_text(row.get(f))

    return meta


claim_meta = {}

# 1. Start from Stage 2 rows.
claim_meta.update(extract_claim_metadata_from_records(stage2_rows))

# 2. Merge Stage 0 metadata if available.
for p in STAGE0_IN_CANDIDATES:
    print("Stage 0 candidate:", p, "exists=", p.exists())
    if p.exists():
        stage0_records = load_json(p)
        stage0_meta = extract_claim_metadata_from_records(stage0_records)
        for cid, m in stage0_meta.items():
            claim_meta.setdefault(cid, {"id": cid, "claim": "", "original_rating": "unknown"})
            if not claim_meta[cid].get("claim") and m.get("claim"):
                claim_meta[cid]["claim"] = m["claim"]
            if claim_meta[cid].get("original_rating") == "unknown" and m.get("original_rating") != "unknown":
                claim_meta[cid]["original_rating"] = m["original_rating"]

# 3. Merge optional claim metadata files if they exist.
for p in CLAIM_METADATA_CANDIDATES:
    if not p.exists():
        continue
    print("Claim metadata candidate found:", p)
    records = load_json(p)
    more_meta = extract_claim_metadata_from_records(records)
    for cid, m in more_meta.items():
        claim_meta.setdefault(cid, {"id": cid, "claim": "", "original_rating": "unknown"})
        if not claim_meta[cid].get("claim") and m.get("claim"):
            claim_meta[cid]["claim"] = m["claim"]
        if claim_meta[cid].get("original_rating") == "unknown" and m.get("original_rating") != "unknown":
            claim_meta[cid]["original_rating"] = m["original_rating"]

if ENFORCE_TARGET_CLAIM_IDS:
    claim_meta = {cid: m for cid, m in claim_meta.items() if cid in TARGET_CLAIM_IDS}

print("\nClaim metadata loaded for IDs:", sorted(claim_meta.keys()))
print("Ratings found:")
for cid in sorted(claim_meta.keys()):
    print(cid, "|", claim_meta[cid].get("original_rating"), "|", claim_meta[cid].get("claim", "")[:100])

missing_ratings = [cid for cid, m in claim_meta.items() if normalize_rating(m.get("original_rating")) == "unknown"]
if missing_ratings:
    print("\nWARNING: original_rating missing/unknown for:", sorted(missing_ratings))
    print("Add your claim metadata path to CLAIM_METADATA_CANDIDATES if needed.")


Stage 0 candidate: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage0/flattened_sources_test_dataset.json exists= True

Claim metadata loaded for IDs: ['0_ambigsnopes', '100_ambigsnopes', '104_ambigsnopes', '106_ambigsnopes', '107_ambigsnopes', '109_ambigsnopes', '111_ambigsnopes', '113_ambigsnopes', '114_ambigsnopes', '116_ambigsnopes', '117_ambigsnopes', '118_ambigsnopes', '11_ambigsnopes', '120_ambigsnopes', '122_ambigsnopes', '124_ambigsnopes', '125_ambigsnopes', '127_ambigsnopes', '128_ambigsnopes', '129_ambigsnopes', '12_ambigsnopes', '130_ambigsnopes', '131_ambigsnopes', '133_ambigsnopes', '134_ambigsnopes', '135_ambigsnopes', '136_ambigsnopes', '139_ambigsnopes', '13_ambigsnopes', '140_ambigsnopes', '142_ambigsnopes', '144_ambigsnopes', '145_ambigsnopes', '146_ambigsnopes', '147_ambigsnopes', '148_ambigsnopes', '149_ambigsnopes', '14_ambigsnopes', '150_ambigsnopes', '151_ambigsnopes', '153_ambigsnopes', '154_ambigsnopes', '157_ambigsnopes', '158_ambigsnopes', '159_am

## 6. Stage 3 selection rules

In [7]:
CORE_ROLES = {"core_support", "core_refutation"}
PROVENANCE_ROLES = {"visual_origin", "visual_misattribution", "image_manipulation"}
ORIGIN_ROLES = {"claim_origin", "satirical_origin", "fake_news_origin"}
CAVEAT_ROLES = {"temporal_refutation", "rating_caveat"}
CONTEXT_ROLES = {"legal_or_definition_context", "scientific_context", "background_context"}
STAT_ROLES = {"statistical_evidence"}
BAD_ROLES = {"off_topic", "no_content"}

MUST_PROMOTE_ROLES = (
    CORE_ROLES
    | PROVENANCE_ROLES
    | {"satirical_origin", "fake_news_origin"}
    | CAVEAT_ROLES
    | STAT_ROLES
)

NUANCE_RATINGS = {"mostly_true", "half_true", "misleading"}
FALSE_RATINGS = {"false", "pants_fire"}


def infer_evidence_role(row):
    role = clean_text(row.get("evidence_role"))
    if role:
        return role

    relation = row.get("source_relation")
    stance = row.get("stance_to_claim")

    if relation == "claim_origin":
        return "claim_origin"
    if relation == "satirical_origin":
        return "satirical_origin"
    if relation == "visual_origin":
        if stance in {"refutes_claim", "refutes_supporting_evidence"}:
            return "visual_misattribution"
        return "visual_origin"
    if relation in {"off_topic", "no_content"}:
        return relation
    if stance == "supports_claim":
        return "core_support"
    if stance == "refutes_claim":
        return "core_refutation"
    if stance == "refutes_supporting_evidence":
        return "visual_misattribution"
    return "background_context"


def has_useful_snippet(row):
    return any(clean_text(s.get("text")) for s in row.get("snippets", []) if isinstance(s, dict))


def row_score(row):
    return safe_int(row.get("claim_match_score"), 0)


def compact_snippets(row):
    snippets = []
    seen = set()
    for snip in row.get("snippets", []):
        if not isinstance(snip, dict):
            continue
        text = clean_text(snip.get("text"), 700)
        if not text or text in seen:
            continue
        seen.add(text)
        snippets.append({
            "snippet_type": clean_text(snip.get("snippet_type")),
            "text": text,
            "stance": clean_text(snip.get("stance")),
            "snippet_score": safe_int(snip.get("snippet_score"), 0),
        })
    snippets.sort(key=lambda x: x.get("snippet_score", 0), reverse=True)
    return snippets[:MAX_SNIPPETS_PER_EVIDENCE]


def make_evidence_item(row, selection_reason):
    role = infer_evidence_role(row)
    return {
        "url": row.get("url", ""),
        "file_name": row.get("file_name", ""),
        "source_relation": clean_text(row.get("source_relation")),
        "evidence_role": role,
        "claim_match_score": row_score(row),
        "stance_to_claim": clean_text(row.get("stance_to_claim")),
        "safe_source_fact": clean_text(row.get("safe_source_fact"), 700),
        "snippets": compact_snippets(row),
        "risk_flags": [clean_text(x) for x in listify(row.get("risk_flags")) if clean_text(x)],
        "caveats": [clean_text(x) for x in listify(row.get("caveats")) if clean_text(x)],
        "selection_reason": selection_reason,
    }


def selection_priority(row):
    role = infer_evidence_role(row)
    score = row_score(row)
    stance = row.get("stance_to_claim")

    if role in CORE_ROLES:
        group = 0
    elif role in PROVENANCE_ROLES or role in {"satirical_origin", "fake_news_origin"}:
        group = 1
    elif role in CAVEAT_ROLES:
        group = 2
    elif role in STAT_ROLES:
        group = 3
    elif role == "claim_origin":
        group = 4
    elif role in CONTEXT_ROLES:
        group = 5
    else:
        group = 6

    stance_bonus = 1 if stance in {"supports_claim", "refutes_claim", "refutes_supporting_evidence", "qualifies_claim", "satire"} else 0
    snippet_bonus = 1 if has_useful_snippet(row) else 0
    return (group, -score, -stance_bonus, -snippet_bonus, clean_text(row.get("url")))


def should_exclude(row):
    role = infer_evidence_role(row)
    relation = row.get("source_relation")
    score = row_score(row)

    if role in BAD_ROLES or relation in {"off_topic", "no_content"}:
        return True, "off-topic or no readable claim-useful content"
    if score <= 1:
        return True, "claim match score too low"
    if not row.get("url"):
        return True, "missing URL"
    if not has_useful_snippet(row) and not clean_text(row.get("safe_source_fact")):
        return True, "no usable snippet or safe source fact"
    return False, ""


def should_must_cite(row, rating):
    role = infer_evidence_role(row)
    relation = row.get("source_relation")
    score = row_score(row)

    if score < 3:
        return False, "score below must-cite threshold"

    if role in CORE_ROLES and score >= 4:
        return True, "core support/refutation evidence"

    if role in PROVENANCE_ROLES and score >= 4:
        return True, "visual provenance or misattribution evidence"

    if role in {"satirical_origin", "fake_news_origin"} and score >= 4:
        return True, "satire/fake-news origin evidence"

    if role in STAT_ROLES and score >= 4:
        return True, "exact statistical/numerical evidence"

    if role in CAVEAT_ROLES and rating in NUANCE_RATINGS and score >= 3:
        return True, "rating caveat or temporal context needed for nuanced verdict"

    if role == "claim_origin" and score >= 3 and has_useful_snippet(row):
        return True, "exact claim-origin quote/post/headline"

    if relation == "direct_evidence" and score >= 4:
        return True, "strong direct evidence"

    return False, "not must-cite by Stage 3 rules"


def should_optional(row):
    role = infer_evidence_role(row)
    score = row_score(row)

    if score >= 3 and role in CONTEXT_ROLES | CAVEAT_ROLES:
        return True, "useful context but not required"
    if score >= 3 and row.get("keep_for_coverage"):
        return True, "coverage-safe supporting context"
    if score == 2 and clean_text(row.get("safe_source_fact")):
        return True, "weak context only"
    return False, "not useful enough for optional context"

## 7. Deduplication and packet building

In [8]:
def dedupe_items(items):
    """Keep strongest item per URL and near-duplicate safe fact."""
    by_url = {}
    for item in items:
        url = item.get("url", "")
        if not url:
            continue
        prev = by_url.get(url)
        if prev is None or item.get("claim_match_score", 0) > prev.get("claim_match_score", 0):
            by_url[url] = item

    deduped = []
    seen_facts = set()
    for item in sorted(by_url.values(), key=lambda x: (x.get("selection_rank", 999), -x.get("claim_match_score", 0))):
        fact_key = clean_text(item.get("safe_source_fact", "")).lower()[:180]
        if fact_key and fact_key in seen_facts:
            continue
        if fact_key:
            seen_facts.add(fact_key)
        item.pop("selection_rank", None)
        deduped.append(item)
    return deduped


def build_verdict_logic(claim, rating, must_items, optional_items, warnings):
    roles = Counter(item.get("evidence_role") for item in must_items + optional_items)
    logic = [f"Preserve the original rating nuance: {rating}."]

    if rating in {"true", "mostly_true"}:
        logic.append("Use core supporting evidence, and include any caveat evidence if the rating is not fully true.")
    elif rating in {"half_true", "misleading"}:
        logic.append("Explain what part of the claim is supported and what context, timing, wording, or caveat changes the interpretation.")
    elif rating in FALSE_RATINGS:
        logic.append("Lead with the strongest refutation or provenance evidence showing why the claim is false.")
    elif rating == "satire":
        logic.append("Make clear that the source/origin is satirical or fictional, especially if the claim was later reused as real news.")
    else:
        logic.append("Use the selected evidence to explain the claim carefully without adding outside facts.")

    if roles.get("visual_misattribution") or roles.get("visual_origin") or roles.get("image_manipulation"):
        logic.append("Include the visual provenance evidence because it may be central to refuting the claimed image/video context.")
    if roles.get("claim_origin"):
        logic.append("Use claim-origin evidence only to show that the claim was made or circulated, not as proof that it is true.")
    if roles.get("rating_caveat") or roles.get("temporal_refutation"):
        logic.append("Use rating-caveat or temporal evidence to preserve verdict nuance.")
    if roles.get("statistical_evidence"):
        logic.append("Use statistical evidence carefully and preserve exact numbers/units.")

    if warnings:
        logic.append("Resolve packet warnings conservatively in Stage 4; do not invent missing evidence.")

    return logic


def validate_packet_requirements(rating, must_items, optional_items):
    roles = Counter(item.get("evidence_role") for item in must_items + optional_items)
    warnings = []

    if not must_items:
        warnings.append("no must_cite_evidence selected")

    if rating in {"true", "mostly_true"} and not roles.get("core_support"):
        warnings.append("rating expects core_support evidence, but none was selected")

    if rating in {"mostly_true", "half_true", "misleading"} and not (roles.get("rating_caveat") or roles.get("temporal_refutation") or roles.get("legal_or_definition_context") or roles.get("background_context")):
        warnings.append("nuanced rating may need caveat/context evidence")

    if rating in FALSE_RATINGS and not (
        roles.get("core_refutation")
        or roles.get("visual_misattribution")
        or roles.get("image_manipulation")
        or roles.get("fake_news_origin")
        or roles.get("temporal_refutation")
    ):
        warnings.append("false/pants_fire rating expects refutation, provenance, fake-origin, or temporal-refutation evidence")

    if rating == "satire" and not roles.get("satirical_origin"):
        warnings.append("satire rating expects satirical_origin evidence")

    if len(must_items) < MIN_MUST_CITE:
        warnings.append(f"must_cite_evidence has fewer than {MIN_MUST_CITE} items")

    if len(must_items) > MAX_MUST_CITE:
        warnings.append(f"must_cite_evidence exceeds cap of {MAX_MUST_CITE} items")

    return warnings


def build_packet_for_claim(cid, rows, meta):
    claim = clean_text(meta.get("claim") or rows[0].get("claim"))
    original_rating_raw = meta.get("original_rating", "unknown")
    rating = normalize_rating(original_rating_raw)

    must_candidates = []
    optional_candidates = []
    excluded = []

    for row in sorted(rows, key=selection_priority):
        exclude, exclude_reason = should_exclude(row)
        if exclude:
            excluded.append(make_evidence_item(row, exclude_reason))
            continue

        must, must_reason = should_must_cite(row, rating)
        if must:
            item = make_evidence_item(row, must_reason)
            item["selection_rank"] = len(must_candidates)
            must_candidates.append(item)
            continue

        optional, optional_reason = should_optional(row)
        if optional:
            item = make_evidence_item(row, optional_reason)
            item["selection_rank"] = len(optional_candidates)
            optional_candidates.append(item)
        else:
            excluded.append(make_evidence_item(row, optional_reason))

    must_items = dedupe_items(must_candidates)
    optional_items = dedupe_items(optional_candidates)

    # If too few must-cite items, promote strongest optional context until the minimum is reached.
    while len(must_items) < MIN_MUST_CITE and optional_items:
        promoted = optional_items.pop(0)
        promoted["selection_reason"] = "promoted from optional to meet compact minimum evidence packet"
        must_items.append(promoted)

    # Enforce must-cite cap. Overflow becomes optional first, then excluded if optional is full.
    if len(must_items) > MAX_MUST_CITE:
        overflow = must_items[MAX_MUST_CITE:]
        must_items = must_items[:MAX_MUST_CITE]
        for item in overflow:
            item["selection_reason"] = "lower-priority overflow after must-cite cap"
            optional_items.append(item)

    optional_items = optional_items[:MAX_OPTIONAL_CONTEXT]

    warnings = validate_packet_requirements(rating, must_items, optional_items)

    all_risk_flags = []
    for row in rows:
        all_risk_flags.extend([clean_text(x) for x in listify(row.get("risk_flags")) if clean_text(x)])
    all_risk_flags = sorted(set(all_risk_flags))

    evidence_summary = {
        "total_stage2_sources": len(rows),
        "must_cite_count": len(must_items),
        "optional_context_count": len(optional_items),
        "excluded_count": len(excluded),
        "roles_in_must_cite": dict(Counter(x.get("evidence_role") for x in must_items)),
        "relations_in_must_cite": dict(Counter(x.get("source_relation") for x in must_items)),
        "stage3_warnings": warnings,
    }

    verdict_logic = build_verdict_logic(claim, rating, must_items, optional_items, warnings)

    return {
        "id": cid,
        "claim": claim,
        "original_rating": original_rating_raw,
        "normalized_rating": rating,
        "verdict_logic": verdict_logic,
        "must_cite_evidence": must_items,
        "optional_context": optional_items,
        "excluded_evidence": excluded,
        "evidence_summary": evidence_summary,
        "risk_flags": all_risk_flags,
        "stage": "stage3_verdict_aware_evidence_packet_builder",
        "run_timestamp_utc": utc_now(),
    }

## 8. Build and save Stage 3 packets

In [9]:
rows_by_claim = defaultdict(list)
for row in stage2_rows:
    cid = claim_id(row)
    if cid:
        rows_by_claim[cid].append(row)

packets = []
for cid in sorted(rows_by_claim.keys()):
    meta = claim_meta.get(cid, {"id": cid, "claim": "", "original_rating": "unknown"})
    packet = build_packet_for_claim(cid, rows_by_claim[cid], meta)
    packets.append(packet)

save_json(STAGE3_OUT, packets)
shutil.copy2(STAGE3_OUT, DRIVE_STAGE3_OUT)

print("Stage 3 packets created:", len(packets))
print("Local output:", STAGE3_OUT)
print("Drive output:", DRIVE_STAGE3_OUT)
print("Drive output exists:", DRIVE_STAGE3_OUT.exists())

Stage 3 packets created: 1146
Local output: /content/outputs/stage3/stage3_packets_test_dataset.json
Drive output: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage3/stage3_packets_test_dataset.json
Drive output exists: True


In [10]:
print("Stage 3 output exists:", DRIVE_STAGE3_OUT.exists())
print("Stage 3 Drive output:", DRIVE_STAGE3_OUT)

packets_check = load_json(DRIVE_STAGE3_OUT)

print("Packet count:", len(packets_check))
print("Unique packet IDs:", len({p["id"] for p in packets_check}))

print("\nFirst 5 packets:")
for p in packets_check[:5]:
    print(
        p["id"],
        "| must:",
        len(p.get("must_cite_evidence", [])),
        "| optional:",
        len(p.get("optional_context", [])),
        "| excluded:",
        len(p.get("excluded_evidence", [])),
        "| rating:",
        p.get("normalized_rating"),
    )

Stage 3 output exists: True
Stage 3 Drive output: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage3/stage3_packets_test_dataset.json
Packet count: 1146
Unique packet IDs: 1146

First 5 packets:
0_ambigsnopes | must: 6 | optional: 1 | excluded: 1 | rating: mostly_false
100_ambigsnopes | must: 5 | optional: 1 | excluded: 1 | rating: half_true
104_ambigsnopes | must: 2 | optional: 2 | excluded: 0 | rating: half_true
106_ambigsnopes | must: 5 | optional: 1 | excluded: 1 | rating: half_true
107_ambigsnopes | must: 2 | optional: 2 | excluded: 2 | rating: half_true


## 9. Summary

In [11]:
for p in packets:
    s = p["evidence_summary"]
    print(
        p["id"],
        "| rating:", p.get("normalized_rating"),
        "| total:", s["total_stage2_sources"],
        "| must:", s["must_cite_count"],
        "| optional:", s["optional_context_count"],
        "| excluded:", s["excluded_count"],
        "| roles:", s["roles_in_must_cite"],
    )
    if s["stage3_warnings"]:
        print("  WARNINGS:", s["stage3_warnings"])

0_ambigsnopes | rating: mostly_false | total: 8 | must: 6 | optional: 1 | excluded: 1 | roles: {'core_support': 3, 'core_refutation': 1, 'claim_origin': 2}
100_ambigsnopes | rating: half_true | total: 7 | must: 5 | optional: 1 | excluded: 1 | roles: {'core_support': 5}
104_ambigsnopes | rating: half_true | total: 4 | must: 2 | optional: 2 | excluded: 0 | roles: {'temporal_refutation': 2}
106_ambigsnopes | rating: half_true | total: 7 | must: 5 | optional: 1 | excluded: 1 | roles: {'core_support': 3, 'core_refutation': 1, 'rating_caveat': 1}
107_ambigsnopes | rating: half_true | total: 6 | must: 2 | optional: 2 | excluded: 2 | roles: {'core_support': 1, 'scientific_context': 1}
109_ambigsnopes | rating: half_true | total: 26 | must: 10 | optional: 4 | excluded: 3 | roles: {'core_refutation': 1, 'core_support': 7, 'visual_origin': 1, 'temporal_refutation': 1}
111_ambigsnopes | rating: half_true | total: 6 | must: 4 | optional: 1 | excluded: 1 | roles: {'core_support': 1, 'visual_misattri

## 10. Inspect one claim

In [12]:
INSPECT_ID = "118_ambigsnopes"

items = [p for p in packets if p["id"] == INSPECT_ID]
if not items:
    print("No packet found for", INSPECT_ID)
else:
    p = items[0]
    print(json.dumps({
        "id": p["id"],
        "claim": p["claim"],
        "original_rating": p["original_rating"],
        "normalized_rating": p["normalized_rating"],
        "verdict_logic": p["verdict_logic"],
        "evidence_summary": p["evidence_summary"],
        "risk_flags": p["risk_flags"],
        "must_cite_evidence": p["must_cite_evidence"],
        "optional_context": p["optional_context"],
    }, ensure_ascii=False, indent=2)[:8000])

{
  "id": "118_ambigsnopes",
  "claim": "Reality TV star Kim Kardashian once flew from Los Angeles to Paris specifically to eat a special kind of cheesecake from a particular hotel.",
  "original_rating": "Mixture",
  "normalized_rating": "half_true",
  "verdict_logic": [
    "Preserve the original rating nuance: half_true.",
    "Explain what part of the claim is supported and what context, timing, wording, or caveat changes the interpretation.",
    "Use claim-origin evidence only to show that the claim was made or circulated, not as proof that it is true.",
    "Resolve packet warnings conservatively in Stage 4; do not invent missing evidence."
  ],
  "evidence_summary": {
    "total_stage2_sources": 11,
    "must_cite_count": 7,
    "optional_context_count": 0,
    "excluded_count": 4,
    "roles_in_must_cite": {
      "core_support": 6,
      "claim_origin": 1
    },
    "relations_in_must_cite": {
      "direct_evidence": 6,
      "claim_origin": 1
    },
    "stage3_warnings": [

## 11. Stage 4 input path

Use this in Stage 4:

```python
STAGE3_IN = DRIVE_BASE / f"outputs/stage3/stage3_packets_{DATASET_TAG}.json"
```